In [ ]:
!pip install tqdm lightgbm
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import lightgbm as lgb
from tqdm import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

csv_path = "/content/drive/MyDrive/amazon/images_test/test.csv"
image_dir = Path("/content/downloaded_images")
image_column = "image_link"
id_column = "sample_id"
MAX_WORKERS = 50
TIMEOUT = 5

image_dir.mkdir(exist_ok=True)
df = pd.read_csv(csv_path)
print(f"Downloading {len(df)} images...")

def download_image(row):
    save_path = image_dir / f"{row[id_column]}.jpg"
    if save_path.exists():
        return row[id_column], True, "cached"
    try:
        r = requests.get(row[image_column], timeout=TIMEOUT)
        r.raise_for_status()
        save_path.write_bytes(r.content)
        return row[id_column], True, None
    except Exception as e:
        return row[id_column], False, str(e)[:50]

successful, failed = [], []
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = {executor.submit(download_image, row): i for i, row in df.iterrows()}
    for future in tqdm(as_completed(futures), total=len(df), desc="Downloading"):
        img_id, ok, err = future.result()
        (successful if ok else failed).append(img_id)

print(f"Done: {len(successful)} ok, {len(failed)} failed")
if failed:
    pd.DataFrame({'sample_id': failed}).to_csv(image_dir / "failed_downloads.csv", index=False)


In [ ]:
# Retry failed image downloads — only needed if failed_downloads.csv was created above
failed_csv = image_dir / "failed_downloads.csv"
if not failed_csv.exists():
    print("No failures to retry")
else:
    original_df = pd.read_csv(csv_path)
    failed_ids = pd.read_csv(failed_csv)[id_column].tolist()
    retry_df = original_df[original_df[id_column].isin(failed_ids)]
    print(f"Retrying {len(retry_df)} images...")

    still_failed = []
    with ThreadPoolExecutor(max_workers=20) as executor:
        futures = {executor.submit(download_image, row): i for i, row in retry_df.iterrows()}
        for future in tqdm(as_completed(futures), total=len(retry_df), desc="Retrying"):
            img_id, ok, err = future.result()
            if not ok:
                still_failed.append((img_id, err))

    print(f"Retry done: {len(retry_df) - len(still_failed)} recovered, {len(still_failed)} still failed")


In [ ]:
import torch
import clip
from PIL import Image

device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess = clip.load("ViT-B/32", device=device)

DRIVE = "/content/drive/MyDrive/amazon/"
TRAIN_CSV = DRIVE + "processed_train.csv"
IMAGE_DIR = "/content/images/images/"
OUTPUT_FILE = DRIVE + "clip_embeddings_log.pkl"

df = pd.read_csv(TRAIN_CSV)
print(f"Loaded {len(df)} products")

embeddings_list, prices_list, ids_list = [], [], []
failed = 0
batch_size = 16
batch_images, batch_texts, batch_data = [], [], []

pbar = tqdm(total=len(df), desc="CLIP extraction")

for _, row in df.iterrows():
    img_path = Path(IMAGE_DIR) / row['image_filename']
    if not img_path.exists():
        failed += 1
        pbar.update(1)
        continue
    try:
        img = Image.open(img_path).convert('RGB')
        text = (str(row.get('title', '')) + ' ' + str(row.get('description', '')))[:500]
        batch_images.append(img)
        batch_texts.append(text)
        batch_data.append({'price': row['price'], 'id': row['id']})
    except Exception:
        failed += 1
        pbar.update(1)
        continue

    if len(batch_images) == batch_size:
        inputs = model.preprocess(text=batch_texts, images=batch_images,
                                   return_tensors="pt", padding=True, truncation=True, max_length=77)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
            combined = np.concatenate([
                outputs.image_embeds.cpu().numpy(),
                outputs.text_embeds.cpu().numpy()
            ], axis=1)
        embeddings_list.append(combined)
        for d in batch_data:
            prices_list.append(d['price'])
            ids_list.append(d['id'])
        batch_images, batch_texts, batch_data = [], [], []
        pbar.update(batch_size)
        torch.cuda.empty_cache()

# Handle remaining
if batch_images:
    inputs = model.preprocess(text=batch_texts, images=batch_images,
                               return_tensors="pt", padding=True, truncation=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        combined = np.concatenate([
            outputs.image_embeds.cpu().numpy(),
            outputs.text_embeds.cpu().numpy()
        ], axis=1)
    embeddings_list.append(combined)
    for d in batch_data:
        prices_list.append(d['price'])
        ids_list.append(d['id'])
    pbar.update(len(batch_images))

pbar.close()

embeddings = np.vstack(embeddings_list)
prices = np.array(prices_list)
ids = np.array(ids_list)
print(f"Extracted {len(embeddings)} embeddings ({failed} failed). Shape: {embeddings.shape}")


In [ ]:
# Log-transform prices to handle skewed distribution before training
prices_log = np.log(prices + 1)  # +1 avoids log(0)

print(f"Price range: ${prices.min():.2f} - ${prices.max():.2f} (mean {prices.mean():.2f})")
print(f"Log range:   {prices_log.min():.4f} - {prices_log.max():.4f}")

data = {
    'embeddings': embeddings,
    'prices': prices,
    'prices_log': prices_log,
    'product_ids': ids,
    'embedding_dim': 1024,
    'model': 'CLIP-ViT-B/32'
}

with open(OUTPUT_FILE, 'wb') as f:
    pickle.dump(data, f)

import shutil
shutil.copy(OUTPUT_FILE, DRIVE)
print(f"Saved to {OUTPUT_FILE}")


In [ ]:
with open(DRIVE + "clip_embeddings_log.pkl", 'rb') as f:
    data = pickle.load(f)

X = np.array(data['embeddings'])
y_log = np.array(data['prices_log'])
y = np.array(data['prices'])
ids = np.array(data['product_ids'])

print(f"X shape: {X.shape}, y shape: {y_log.shape}")

X_train, X_val, y_train, y_val = train_test_split(X, y_log, test_size=0.1, random_state=42)

params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 64,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'device': 'gpu',
    'gpu_use_dp': False,
    'max_bin': 63,
    'verbose': -1,
}

train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

lgb_model = lgb.train(
    params,
    train_data,
    valid_sets=[train_data, val_data],
    valid_names=['train', 'valid'],
    num_boost_round=10000,
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

print(f"Best iteration: {lgb_model.best_iteration}, best score: {lgb_model.best_score}")

model_save_path = DRIVE + "lgb_model.txt"
lgb_model.save_model(model_save_path)
print(f"Model saved to {model_save_path}")


In [ ]:
import warnings
from transformers import CLIPProcessor, CLIPModel

warnings.filterwarnings("ignore", category=UserWarning)
Image.MAX_IMAGE_PIXELS = None

test_csv_path = "/content/drive/MyDrive/amazon/images_test/test.csv"
test_image_dir = Path("/content/downloaded_images")
output_path = Path("/content/clip_combined_embeddings_test.pkl")
id_column = "sample_id"
text_column = "catalog_content"
MODEL_NAME = "openai/clip-vit-base-patch32"
BATCH_SIZE = 128

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
processor = CLIPProcessor.from_pretrained(MODEL_NAME, use_fast=True)

df_test = pd.read_csv(test_csv_path)
df_test[text_column] = df_test[text_column].fillna("").astype(str)
df_test['image_path'] = df_test[id_column].apply(lambda x: test_image_dir / f"{x}.jpg")
df_test = df_test[df_test['image_path'].apply(lambda p: p.exists())]
print(f"Processing {len(df_test)} test items")

all_ids, all_image_embs, all_text_embs = [], [], []

for i in tqdm(range(0, len(df_test), BATCH_SIZE), desc="Generating embeddings"):
    batch = df_test.iloc[i:i+BATCH_SIZE]
    batch_images, valid_ids = [], []
    for _, row in batch.iterrows():
        try:
            batch_images.append(Image.open(row['image_path']).convert("RGB"))
            valid_ids.append(row[id_column])
        except Exception:
            continue
    if not batch_images:
        continue
    batch_texts = batch[text_column].tolist()[:len(batch_images)]
    inputs = processor(text=batch_texts, images=batch_images,
                       return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        out = clip_model(**inputs)
        img_embs = out.image_embeds
        txt_embs = out.text_embeds
    img_embs /= img_embs.norm(dim=-1, keepdim=True)
    txt_embs /= txt_embs.norm(dim=-1, keepdim=True)
    all_ids.extend(valid_ids)
    all_image_embs.append(img_embs.cpu().numpy())
    all_text_embs.append(txt_embs.cpu().numpy())

combined = np.concatenate((np.vstack(all_image_embs), np.vstack(all_text_embs)), axis=1)
output = {'sample_ids': np.array(all_ids), 'combined_embeddings': combined}

with open(output_path, 'wb') as f:
    pickle.dump(output, f)

import shutil
shutil.copy(str(output_path), DRIVE)
print(f"Saved {len(all_ids)} embeddings, shape {combined.shape}")


In [ ]:
with open("/content/clip_combined_embeddings_test.pkl", 'rb') as f:
    test_data = pickle.load(f)

X_test = test_data["combined_embeddings"]
sample_ids = test_data["sample_ids"]

bst = lgb.Booster(model_file=DRIVE + "lgb_model.txt")
log_preds = bst.predict(X_test)
predicted_prices = np.expm1(log_preds).clip(0)

results_df = pd.DataFrame({'sample_id': sample_ids, 'price': predicted_prices})
results_df.to_csv(DRIVE + "price_predictions.csv", index=False)

print(f"Saved {len(results_df)} predictions")
print(results_df.head())
